In [1]:
! rm *.csv

In [2]:
import subprocess
from pathlib import Path

import pandas as pd

for csv_file in Path('.').glob('*.csv'):
    csv_file.unlink()

def append_queries(queries, output_csv):
    frames = []
    for query in queries:
        result = subprocess.run(query, check=True, capture_output=True, text=True)
        csv_name = next(
            line.strip()
            for line in reversed(result.stdout.splitlines())
            if line.strip().endswith('.csv')
        )
        frames.append(pd.read_csv(csv_name))
        Path(csv_name).unlink()
    if frames:
        df = pd.concat(frames, ignore_index=True)
        header = not Path(output_csv).exists()
        df.to_csv(output_csv, mode='a', header=header, index=False)


In [3]:
computation_queries = [
    [
        '../bin/benchpark', 'query',
        '../wkp/rocm720-tioga/amg2023/', '../wkp/rocm642-tioga/amg2023/',
        '../wkp/rocm720-frontier/amg2023/', '../wkp/rocm642-frontier/amg2023/',
        '--query-regions-byname', 'Problem',
        '--metric', 'Avg time/rank (exc)',
        '--metadata-columns', 'packages.dependencies.hip.version',
        '--exclude-regions', 'MPI_',
    ],
    [
        '../bin/benchpark', 'query',
        '../wkp/rocm720-tioga/kripke/', '../wkp/rocm642-tioga/kripke/',
        '../wkp/rocm720-frontier/kripke/', '../wkp/rocm642-frontier/kripke/',
        '--query-regions-byname', 'main',
        '--metric', 'Avg time/rank (exc)',
        '--metadata-columns', 'packages.dependencies.hip.version',
        '--exclude-regions', 'MPI_',
    ],
    [
        '../bin/benchpark', 'query',
        '../wkp/rocm720-tioga/laghos/', '../wkp/rocm642-tioga/laghos/',
        '../wkp/rocm720-frontier/laghos/', '../wkp/rocm642-frontier/laghos/',
        '--query-regions-byname', 'main',
        '--metric', 'Avg time/rank (exc)',
        '--metadata-columns', 'packages.dependencies.hip.version',
        '--exclude-regions', 'MPI_',
    ],
]

append_queries(computation_queries, 'computation-time.csv')

In [4]:
communication_queries = [
    [
        '../bin/benchpark', 'query',
        '../wkp/rocm720-tioga/amg2023/', '../wkp/rocm642-tioga/amg2023/',
        '../wkp/rocm720-frontier/amg2023/', '../wkp/rocm642-frontier/amg2023/',
        '--query-regions-byname', 'Problem',
        '--filter-regions-byname', 'MPI_',
        '--metric', 'Avg time/rank (exc)',
        '--metadata-columns', 'packages.dependencies.hip.version',
    ],
    [
        '../bin/benchpark', 'query',
        '../wkp/rocm720-tioga/kripke/', '../wkp/rocm642-tioga/kripke/',
        '../wkp/rocm720-frontier/kripke/', '../wkp/rocm642-frontier/kripke/',
        '--filter-regions-byname', 'MPI_',
        '--metric', 'Avg time/rank (exc)',
        '--metadata-columns', 'packages.dependencies.hip.version',
    ],
    [
        '../bin/benchpark', 'query',
        '../wkp/rocm720-tioga/laghos/', '../wkp/rocm642-tioga/laghos/',
        '../wkp/rocm720-frontier/laghos/', '../wkp/rocm642-frontier/laghos/',
        '--filter-regions-byname', 'MPI_',
        '--metric', 'Avg time/rank (exc)',
        '--metadata-columns', 'packages.dependencies.hip.version',
    ],
]

append_queries(communication_queries, 'communication-time.csv')

In [5]:
! ls *.csv

communication-time.csv computation-time.csv
